In [17]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import statsmodels.api as sm


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)

## Feature engineering & Data preprocessing

In [80]:
from sklearn.preprocessing import StandardScaler


def scale_data(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_scaled = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
    X_test_scaled = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
    return X_train_scaled, X_test_scaled, scaler

In [81]:
def prepare_index(df):
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df.asfreq('D')
    df = df.ffill() 
    return df

In [ ]:
def create_features(df):
    df = df.copy()
    df["target"] = df["temperature_2m_mean"].shift(-1)
    for lag in [1, 2, 3, 7, 14, 30]:
        df[f"lag_{lag}"] = df["temperature_2m_mean"].shift(lag)
    for w in [3, 7, 14]:
        df[f"roll_mean_{w}"] = df["temperature_2m_mean"].shift(1).rolling(w).mean()
    df["diff_1"] = df["temperature_2m_mean"].diff(1)
    df["day_of_year"] = df.index.dayofyear
    df["month"] = df.index.month
    df["weekday"] = df.index.weekday
    df = df.dropna()
    X = df.drop(columns=["target"])
    y = df["target"]
    return X, y


def create_features_improved(df):
    df = df.copy()
    df['cos_day'] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)
    df['sin_day'] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
    df['cos_month'] = np.cos(2 * np.pi * df.index.month / 12)
    df['sin_month'] = np.sin(2 * np.pi * df.index.month / 12)
    df['cos_weekday'] = np.cos(2 * np.pi * df.index.weekday / 7)
    df['sin_weekday'] = np.sin(2 * np.pi * df.index.weekday / 7)
    
    df["target"] = df["temperature_2m_mean"].shift(-1)
    df["diff_1"] = df["temperature_2m_mean"].diff(1)
    
    for lag in [1, 7, 365]: 
        if len(df) > lag:
            df[f"lag_{lag}"] = df["temperature_2m_mean"].shift(lag)
    # for w in [3, 7, 14]:
    #     df[f"roll_mean_{w}"] = df["temperature_2m_mean"].shift(1).rolling(w).mean()
    df = df.dropna()
    return df.drop(columns=["target"]), df["target"]


def create_features_final(df):
    df = df.copy()
    df["target"] = df["temperature_2m_mean"].shift(-1)
    
    time_periods = {
        "day_of_year": 365.25,
        "month": 12,
        "weekday": 7
    }
    for col, period in time_periods.items():
        if col == "day_of_year": values = df.index.dayofyear
        elif col == "month": values = df.index.month
        else: values = df.index.weekday
        
        df[f"{col}_sin"] = np.sin(2 * np.pi * values / period)
        df[f"{col}_cos"] = np.cos(2 * np.pi * values / period)

    for w in [3, 7, 14]:
        df[f"roll_mean_{w}"] = df["temperature_2m_mean"].rolling(w).mean()
        df[f"roll_std_{w}"] = df["temperature_2m_mean"].rolling(w).std()

    df["diff_1"] = df["temperature_2m_mean"].diff(1)
    # df["day_of_year"] = df.index.dayofyear
    # df["month"] = df.index.month
    # df["weekday"] = df.index.weekday
    for lag in [1, 2, 3, 7, 14, 30]:
        df[f"lag_{lag}"] = df["temperature_2m_mean"].shift(lag)
        
    df = df.dropna()
    return df.drop(columns=["target"]), df["target"]

## Metrics

In [19]:
def metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

## XGBoost

In [49]:
def train_xgb(X, y):
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        model = XGBRegressor(
            n_estimators=200,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        metrics_ = metrics(y_test, preds)
        print(fold_idx, ", ".join([f"{m}: {s}" for m, s in metrics_.items()]))
        scores.append(metrics_)
    return pd.DataFrame(scores).mean()

## SARIMA

In [42]:
from statsmodels.tsa.deterministic import DeterministicProcess


def train_sarima(series):
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    for train_idx, test_idx in tscv.split(series):
        train, test = series.iloc[train_idx], series.iloc[test_idx]
        model = sm.tsa.statespace.SARIMAX(
            train,
            order=(2, 1, 2),
            seasonal_order=(1, 1, 1, 365),
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        res = model.fit(disp=False)
        preds = res.forecast(len(test))
        scores.append(metrics(test, preds))
    return pd.DataFrame(scores).mean()


def train_sarimax(df: pd.DataFrame):
    y = df["temperature_2m_mean"]
    
    # Создаем гармоники Фурье для годовой сезонности (K=5-10 обычно достаточно)
    fourier = DeterministicProcess(
        y.index, constant=True, period=365.25, fourier=5
    ).in_sample()
    
    # exog = df[[
    #     "month",
    #     "day_of_year",
    #     "mean_pressure",
    #     "humidity"
    # ]]
    exog = df.drop(columns=["temperature_2m_mean"])
    
    # Объединяем с вашими exog (давление, влажность)
    exog = pd.concat([fourier, exog], axis=1)
    
    # Обучаем простую модель ARIMAX без внутреннего seasonal_order
    # model = sm.tsa.statespace.SARIMAX(
    #     y, exog=exog, order=(2, 1, 2)
    # )
    
    tscv = TimeSeriesSplit(n_splits=5)
    scores = []
    
    for train_idx, test_idx in tscv.split(y):
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
        X_train, X_test = exog.iloc[train_idx], exog.iloc[test_idx]
        
        model = sm.tsa.statespace.SARIMAX(
            y_train,
            exog=X_train,
            order=(2, 1, 2),
            # seasonal_order=(1, 1, 1, 365)
        )
        
        res = model.fit(disp=False)
        
        preds = res.forecast(len(y_test), exog=X_test)
        
        scores.append(metrics(y_test, preds))
    
    return pd.DataFrame(scores).mean()

## Experiment

In [22]:
def run_benchmark():
    set_seed(42)

    df = pd.read_csv("data_2023_2026.csv", parse_dates=["time"], date_format="%Y-%m-%d")
    df = df.sort_values("time")
    df = df.set_index("time")
    df["temperature_2m_mean"] = df["temperature_2m_mean (°C)"]
    df = df.drop(columns=["temperature_2m_mean (°C)"])

    if "temperature_2m_mean" not in df.columns:
        raise ValueError("No column temperature_2m_mean")

    print(f"Dataset size: {df.shape}")
    X, y = create_features(df)
    print(f"After feature engineering: {X.shape}")

    print("\nTraining XGBoost...")
    xgb_scores = train_xgb(X, y)
    print("XGBoost metrics:")
    print(xgb_scores)

    print("\nTraining SARIMA...")
    sarima_scores = train_sarimax(df)
    print("SARIMA metrics:")
    print(sarima_scores)

    print("\n=== FINAL COMPARISON ===")
    results = pd.DataFrame({"XGBoost": xgb_scores, "SARIMA": sarima_scores})
    print(results)
    return results

In [23]:
results = run_benchmark()

Dataset size: (1096, 12)
After feature engineering: (1065, 25)

Training XGBoost...
XGBoost metrics:
MAE     3.017874
RMSE    4.043682
R2      0.528654
dtype: float64

Training SARIMA...


c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for ARMA and trend. All parameters except for variances will be set to zeros.
  warn('Too few observations to estimate starting parameters%s.'
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\statespace\sarimax.py:866: UserWarning: Too few observations to estimate starting parameters for seasonal ARMA. 

MemoryError: Unable to allocate 769. MiB for an array with shape (734, 734, 187) and data type float64

In [26]:
df = pd.read_csv("data_2023_2026.csv", parse_dates=["time"], date_format="%Y-%m-%d")
df = df.sort_values("time")
df = df.set_index("time")
df["temperature_2m_mean"] = df["temperature_2m_mean (°C)"]
df = df.drop(columns=["temperature_2m_mean (°C)"])

In [27]:
df.columns

Index(['apparent_temperature_max (°C)', 'apparent_temperature_min (°C)',
       'precipitation_sum (mm)', 'rain_sum (mm)', 'snowfall_sum (cm)',
       'wind_speed_10m_max (km/h)', 'wind_gusts_10m_max (km/h)',
       'shortwave_radiation_sum (MJ/m²)', 'sunshine_duration (s)',
       'relative_humidity_2m_mean (%)', 'pressure_msl_mean (hPa)',
       'temperature_2m_mean'],
      dtype='object')

In [78]:
print(f"Dataset size: {df.shape}")
X, y = create_features_final(df)
print(f"After feature engineering: {X.shape}")

Dataset size: (1096, 12)
After feature engineering: (1065, 31)


In [79]:
print("\nTraining XGBoost...")
xgb_scores = train_xgb(X, y)
print("XGBoost metrics:")
print(xgb_scores)


Training XGBoost...
0 MAE: 4.227361854035302, RMSE: 6.372195464903963, R2: 0.4361099134171422
1 MAE: 1.6923959883677084, RMSE: 2.1145042987922578, R2: 0.8695765961605769
2 MAE: 1.9062771639812968, RMSE: 2.708225839558302, R2: 0.6455891353375849
3 MAE: 1.5606269105006074, RMSE: 2.036139544983643, R2: 0.8671837533824776
4 MAE: 2.1411692455600377, RMSE: 2.8635657392611202, R2: 0.8538433829524018
XGBoost metrics:
MAE     2.305566
RMSE    3.218926
R2      0.734461
dtype: float64


In [45]:
print("\nTraining SARIMA...")
sarima_scores = train_sarimax(df)
print("SARIMA metrics:")
print(sarima_scores)


Training SARIMA...


c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency D will be used.
  self._init_dates(dates, freq)
c:\Users\Максим\AppData\Local\Progra

SARIMA metrics:
MAE      29.282346
RMSE     38.351799
R2     -118.258815
dtype: float64


## Parameter optimization

In [ ]:
import optuna
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error


def objective_xgb(trial, X, y):
    tscv = TimeSeriesSplit(n_splits=3)
    maes = []

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 1.0, log=True),
    }

    for train_idx, val_idx in tscv.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

        model = XGBRegressor(**params, random_state=42)
        model.fit(X_train, y_train)
        preds = model.predict(X_val)
        maes.append(mean_absolute_error(y_val, preds))

    return np.mean(maes)


study = optuna.create_study(study_name="xgb_opt", direction="minimize")
study.optimize(lambda trial: objective_xgb(trial, X, y), n_trials=50)

print(f"Best MAE: {study.best_value}")
print(f"Best params: {study.best_params}")

[I 2026-04-28 12:15:09,784] A new study created in memory with name: no-name-0f4b1f03-e6e8-45f8-966f-556421cc76b8
[I 2026-04-28 12:15:15,576] Trial 0 finished with value: 1.9592006195252551 and parameters: {'n_estimators': 333, 'max_depth': 9, 'learning_rate': 0.19828354228460296, 'subsample': 0.7363902551281901, 'colsample_bytree': 0.5250103804456431, 'gamma': 4.102516878244311e-07, 'reg_alpha': 0.0012385829088778703, 'reg_lambda': 3.4960501634084095e-05}. Best is trial 0 with value: 1.9592006195252551.
[I 2026-04-28 12:15:18,657] Trial 1 finished with value: 1.7524461902668678 and parameters: {'n_estimators': 161, 'max_depth': 7, 'learning_rate': 0.03268986149801162, 'subsample': 0.5090784921868966, 'colsample_bytree': 0.6600902909601722, 'gamma': 0.0011170609138684445, 'reg_alpha': 0.00015395349173395406, 'reg_lambda': 0.017157390382077486}. Best is trial 1 with value: 1.7524461902668678.
[I 2026-04-28 12:15:24,360] Trial 2 finished with value: 1.8573720567768677 and parameters: {'n

Best MAE: 1.6709024915410986
Best params: {'n_estimators': 311, 'max_depth': 4, 'learning_rate': 0.023506490656537968, 'subsample': 0.7182725260375129, 'colsample_bytree': 0.5325497994560449, 'gamma': 0.08778160657947483, 'reg_alpha': 0.0010030067442366119, 'reg_lambda': 0.19684220847568648}


In [ ]:
import pandas as pd
import numpy as np
import optuna
import statsmodels.api as sm
from statsmodels.tsa.deterministic import DeterministicProcess
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

def prepare_index(df):
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df = df.asfreq('D')
    df = df.ffill() 
    return df

def objective_sarimax(trial, df):
    p = trial.suggest_int("p", 0, 2)
    d = trial.suggest_int("d", 0, 1)
    q = trial.suggest_int("q", 0, 2)
    
    fourier_k = trial.suggest_int("fourier_k", 0, 10)
    tscv = TimeSeriesSplit(n_splits=5)
    maes = []
    
    for train_idx, val_idx in tscv.split(df):
        train_data = df.iloc[train_idx]
        val_data = df.iloc[val_idx]
        
        y_train = train_data["temperature_2m_mean"]
        y_val = val_data["temperature_2m_mean"]
        
        dp = DeterministicProcess(
            train_data.index, 
            period=365.25, 
            fourier=fourier_k
        )
        fourier_train = dp.in_sample()
        exog_train = pd.concat([train_data.drop(columns=["temperature_2m_mean"]), fourier_train], axis=1)
        
        fourier_val = dp.out_of_sample(steps=len(val_data))
        exog_val = pd.concat([val_data.drop(columns=["temperature_2m_mean"]), fourier_val], axis=1)
        
        scaler = StandardScaler()
        exog_train_scaled = scaler.fit_transform(exog_train)
        exog_val_scaled = scaler.transform(exog_val)
        
        try:
            model = sm.tsa.statespace.SARIMAX(
                y_train,
                exog=exog_train_scaled,
                order=(p, d, q),
                enforce_stationarity=False,
                enforce_invertibility=False
            ).fit(disp=False, maxiter=50)
            
            preds = model.forecast(steps=len(y_val), exog=exog_val_scaled)
            maes.append(mean_absolute_error(y_val, preds))
            
        except Exception as e:
            #  If the model did not converge - return a big error
            return 1e9

    return np.mean(maes)

indexed_df = prepare_index(df)
study = optuna.create_study(study_name="sarima_opt", direction="minimize")
study.optimize(lambda trial: objective_sarimax(trial, indexed_df), n_trials=30)

print(f"Best MAE: {study.best_value}")
print(f"Best params: {study.best_params}")

[I 2026-04-28 15:07:05,972] A new study created in memory with name: sarima_opt
[I 2026-04-28 15:07:13,287] Trial 0 finished with value: 0.8249953298607945 and parameters: {'p': 1, 'd': 1, 'q': 0, 'fourier_k': 0}. Best is trial 0 with value: 0.8249953298607945.
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
[I 2026-04-28 15:07:41,020] Trial 1 finished with value: 13595.994880090473 and parameters: {'p': 1, 'd': 0, 'q': 2, 'fourier_k': 9}. Best is trial 0 with value: 0.8249953298607945.
c:\Users\Максим\AppData\Local\Programs\Python\Python312\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\Максим\AppData\Local\Programs\Pytho

Best MAE: 0.6773572961458133
Best params: {'p': 2, 'd': 1, 'q': 2, 'fourier_k': 1}


## Direct parameter optimization for SARIMA

In [ ]:
from statsmodels.tsa.stattools import adfuller

result = adfuller(df['temperature_2m_mean'])
print(f'p-value: {result[1]}') 